# 核心实现文件总览
##### verl 的 NCCL 重分片机制分布在以下几个关键文件中：

In [ ]:
verl/
├── workers/
│   ├── sharding_manager/
│   │   ├── megatron_vllm.py          # ★ MegatronVLLMShardingManager (主实现)
│   │   ├── megatron_sglang.py       # Megatron-SGLang 版本
│   │   ├── fsdp_vllm.py             # FSDP-vLLM 版本
│   │   └── fsdp_ulysses.py          # FSDP-Ulysses 版本
│   ├── megatron_workers.py          # ★ ActorRolloutRefWorker 调用入口
│   └── actor/
│       └── megatron_actor.py        # MegatronPPOActor 训练逻辑
├── third_party/
│   └── vllm/
│       ├── vllm_v_0_6_3/
│       │   ├── parallel_state.py     # ★ vLLM 分布式环境初始化
│       │   ├── megatron_weight_loaders.py  # ★ Megatron→vLLM 权重加载
│       │   └── dtensor_weight_loaders.py   # FSDP→vLLM 权重加载
│       └── vllm_spmd/               # vLLM >= 0.7 适配器
├── models/
│   └── mcore/                       # Megatron-Core 模型实现
│       └── weight_loader_registery.py  # 权重加载注册表
└── utils/
    └── megatron_utils.py            # Megatron 工具函数 (offload 等)

# 核心文件详解
## 1. verl/workers/sharding_manager/megatron_vllm.py ★ 最重要
这是 3D-HybridEngine 的核心实现文件，包含 MegatronVLLMShardingManager 和 AllGatherPPModel 两个关键类。
- https://github.com/verl-project/verl/tree/v0.6.1/verl/workers/sharding_manager
### 关键类：

| 类名                            | 职责                                        | 代码行数       |
| ----------------------------- | ----------------------------------------- | ---------- |
| `AllGatherPPModel`            | 维护 Actor 和 Rollout 的双缓冲内存，支持 PP 维度权重聚合    | ~200-300 行 |
| `MegatronVLLMShardingManager` | Context Manager，执行 Megatron ↔ vLLM 的权重重分片 | ~400-500 行 |

### MegatronVLLMShardingManager 的核心方法：

In [ ]:
# 文件: verl/workers/sharding_manager/megatron_vllm.py

class MegatronVLLMShardingManager(BaseShardingManager):
    """
    Context Manager: 进入时 Megatron→vLLM, 退出时 vLLM→Megatron
    """
    
    def __init__(self, 
                 module,                    # AllGatherPPModel 实例
                 inference_engine,          # vLLM LLMEngine
                 model_config,
                 transformer_config,
                 layer_name_mapping,        # Megatron↔vLLM 层名映射
                 weight_converter=None,
                 actor_module=None):
        self.module = module
        self.inference_engine = inference_engine
        self.model_config = model_config
        self.transformer_config = transformer_config
        self.layer_name_mapping = layer_name_mapping
        self.weight_converter = weight_converter
        self.actor_module = actor_module
    
    def __enter__(self):
        """
        训练 → 生成 转换:
        1. per_tensor_generator: 遍历 Megatron 模型参数
        2. all_gather PP 维度
        3. 层名映射 + TP 重分片
        4. load_weights 到 vLLM
        """
        # 关键: 逐张量生成器，处理 Megatron→vLLM 权重转换
        per_tensor_param = per_tensor_generator(
            self.actor_module,
            self.model_config,
            self.weight_converter,
            self.transformer_config,
            self.layer_name_mapping,
        )
        
        # 加载到 vLLM 推理引擎
        model = self.model_runner.model
        loaded_params = model.load_weights(per_tensor_param)
        
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        """
        生成 → 训练 恢复:
        1. vLLM sleep (释放 KV Cache)
        2. 恢复 Megatron 训练状态
        """
        self.inference_engine.sleep(level=VLLM_SLEEP_LEVEL)
        for model in self.actor_module:
            model.train()

### per_tensor_generator 函数（<font color='red'>关键的重分片逻辑</font>）：

In [ ]:
# 文件: verl/workers/sharding_manager/megatron_vllm.py

from verl.utils.megatron_utils import load_megatron_model_to_gpu, offload_megatron_model_to_cpu, per_tensor_generator

def per_tensor_generator(actor_module, model_config, weight_converter,
                         transformer_config, layer_name_mapping):
    """
    逐张量遍历 Megatron 模型参数，转换为 vLLM 格式
    """
    for model in actor_module:
        for name, param in model.named_parameters():
            # 1. 获取 Megatron 格式的参数 (已按 TP/PP 分片)
            megatron_param = param
            
            # 2. AllGather PP 维度 (在 micro DP 组内)
            # AllGatherPPModel 内部实现
            gathered_param = all_gather_pp_param(megatron_param)
            
            # 3. 层名映射: Megatron 层名 → vLLM 层名
            vllm_name = layer_name_mapping.get(name, name)
            
            # 4. 权重格式转换 (如 QKV 重排)
            if weight_converter is not None:
                converted_param = weight_converter(gathered_param, vllm_name)
            else:
                converted_param = gathered_param
            
            # 5. 按 vLLM 的 TP 方式重新分片
            vllm_shard = shard_for_vllm_tp(converted_param, vllm_tp_size)
            
            yield vllm_name, vllm_shard

## verl/workers/megatron_workers.py — 调用入口
这是 Worker 的实现文件，ActorRolloutRefWorker 在这里初始化 HybridEngine 并在 generate_sequences 中调用重分片。
### 关键代码段：

In [ ]:
# 文件: verl/workers/megatron_workers.py (约第 63 行起)

class ActorRolloutRefWorker(MegatronWorker):
    
    @register(dispatch_mode=Dispatch.ONE_TO_ALL)
    def init_model(self):
        """
        初始化 HybridEngine、Actor、Rollout
        """
        # 1. 初始化 3D HybridEngine
        self.hybrid_engine = AllGatherPPModel(
            model_provider=megatron_actor_model_provider
        )
        # 获取当前 rank 的模型
        self.actor_module = self.hybrid_engine.this_rank_models
        
        # 2. 构建 Actor 训练模型
        self.actor = MegatronPPOActor(
            config=self.config.actor,
            model_config=self.actor_model_config,
            megatron_config=megatron_config,
            actor_module=self.actor_module,
            actor_optimizer=self.actor_optimizer,
            actor_optimizer_config=self.actor_optim_config
        )
        
        # 3. 构建 Rollout (vLLM)
        rollout = vLLMRollout(
            actor_module=params,
            config=self.config.rollout,
            tokenizer=self.tokenizer,
            model_hf_config=self.actor_model_config,
            train_tp=mpu.get_tensor_model_parallel_world_size()
        )
        
        # 4. ★ 创建 ShardingManager (NCCL 重分片核心)
        self.sharding_manager = MegatronVLLMShardingManager(
            module=self.hybrid_engine,
            inference_engine=rollout.inference_engine,
            model_config=self.actor_model_config,
            layer_name_mapping=layer_name_mapping
        )
    
    @register(dispatch_mode=Dispatch.MEGATRON_PP_AS_DP_PROTO)
    def generate_sequences(self, prompts: DataProto):
        """
        生成序列: 触发 NCCL 重分片 → vLLM 生成 → 恢复训练布局
        """
        with self.sharding_manager:  # ★ 进入: Megatron→vLLM, 退出: vLLM→Megatron
            # 此时权重已重分片到 vLLM 布局
            sequences = self.rollout.generate(prompts)
            # Actor 计算 old_log_prob
            old_log_probs = self.actor.compute_log_prob(sequences)
        
        # 上下文退出，自动恢复 Megatron 训练布局
        return sequences, old_log_probs

## verl/third_party/vllm/vllm_v_0_6_3/parallel_state.py — 双分布式环境
关键设计：Megatron 和 vLLM 各自维护独立的分布式环境，避免全局变量冲突。


In [ ]:
# 文件: verl/third_party/vllm/vllm_v_0_6_3/parallel_state.py

import megatron.core.parallel_state as mpu  # Megatron 的并行状态
import vllm.parallel_state as vllm_ps      # vLLM 的并行状态

def initialize_dual_parallel_state(megatron_tp, megatron_pp, megatron_dp,
                                   vllm_tp, vllm_dp):
    """
    同时初始化两套分布式环境:
    - mpu: Megatron 的 TP/PP/DP 组
    - vllm_ps: vLLM 的 TP/DP 组
    """
    # Megatron 环境
    mpu.initialize_model_parallel(
        tensor_model_parallel_size=megatron_tp,
        pipeline_model_parallel_size=megatron_pp,
        data_parallel_size=megatron_dp
    )
    
    # vLLM 环境 (独立命名空间)
    vllm_ps.initialize_parallel_state(
        tensor_model_parallel_size=vllm_tp,
        data_parallel_size=vllm_dp
    )
    
    # 两套 group 互不干扰，通过 rank 映射进行通信

## verl/third_party/vllm/vllm_v_0_6_3/megatron_weight_loaders.py — 权重加载器
处理 Megatron 格式 → vLLM 格式的具体转换：


In [ ]:
# 文件: verl/third_party/vllm/vllm_v_0_6_3/megatron_weight_loaders.py

def load_megatron_weights_to_vllm(megatron_params, vllm_model, 
                                   layer_name_mapping):
    """
    将 Megatron 格式的权重加载到 vLLM 模型
    """
    for megatron_name, param in megatron_params:
        # 1. 层名转换
        vllm_name = layer_name_mapping[megatron_name]
        
        # 2. 特殊处理: QKV 权重重排
        if 'qkv' in vllm_name:
            # Megatron: [Q0,K0,V0, Q1,K1,V1, ...] (按 attention group 穿插)
            # vLLM: [Q,Q,Q,..., K,K,K,..., V,V,V,...] (按类型聚合)
            param = reshape_qkv_for_vllm(param)
        
        # 3. MLP gate+up 合并
        elif 'gate_up' in vllm_name:
            gate, up = split_gate_up(param)
            param = torch.cat([gate, up], dim=0)
        
        # 4. 加载到 vLLM
        vllm_model.load_parameter(vllm_name, param)

## verl/utils/megatron_utils.py — Offload 工具
内存管理相关：

In [ ]:
# 文件: verl/utils/megatron_utils.py (约 240-420 行)

def offload_megatron_model_to_cpu(model):
    """将 Megatron 模型参数卸载到 CPU"""
    for param in model.parameters():
        param.data = param.data.cpu()

def load_megatron_model_to_gpu(model):
    """将 Megatron 模型参数加载回 GPU"""
    for param in model.parameters():
        param.data = param.data.cuda()

def offload_megatron_optimizer(optimizer):
    """卸载优化器状态"""
    for state in optimizer.state.values():
        for key, value in state.items():
            if isinstance(value, torch.Tensor):
                state[key] = value.cpu()

# NCCL 重分片的调用链

In [ ]:
Ray Driver (single controller)
    │
    ▼
generate_sequences(prompts)  @ MEGATRON_PP_AS_DP_PROTO
    │
    ▼
ActorRolloutRefWorker (每个 GPU 一个 Ray Actor)
    │
    ▼
with self.sharding_manager:     ← 进入上下文
    │
    ├── MegatronVLLMShardingManager.__enter__()
    │       │
    │       ├── per_tensor_generator()
    │       │       │
    │       │       ├── 遍历 actor_module.named_parameters()
    │       │       │
    │       │       ├── all_gather_pp_param()  ← NCCL all_gather
    │       │       │       │
    │       │       │       └── AllGatherPPModel.all_gather()
    │       │       │               │
    │       │       │               └── dist.all_gather_into_tensor()  ← ★ NCCL
    │       │       │
    │       │       ├── layer_name_mapping (Megatron→vLLM 名称转换)
    │       │       │
    │       │       ├── weight_converter (QKV 重排等格式转换)
    │       │       │
    │       │       └── shard_for_vllm_tp() (按 vLLM TP 重新分片)
    │       │
    │       └── model.load_weights() (加载到 vLLM 引擎)
    │
    ├── vLLM.generate() (自回归生成)
    │
    ├── Actor.compute_log_prob() (计算 old log prob)
    │
    └── MegatronVLLMShardingManager.__exit__()
            │
            ├── inference_engine.sleep() (释放 vLLM KV Cache)
            │
            └── model.train() (恢复 Megatron 训练模式)

# 关键代码行号参考

| 文件                                                              | 关键代码段                                   | 约略行号     |
| --------------------------------------------------------------- | --------------------------------------- | -------- |
| `verl/workers/sharding_manager/megatron_vllm.py`                | `MegatronVLLMShardingManager.__enter__` | ~150-200 |
| `verl/workers/sharding_manager/megatron_vllm.py`                | `MegatronVLLMShardingManager.__exit__`  | ~200-250 |
| `verl/workers/sharding_manager/megatron_vllm.py`                | `per_tensor_generator`                  | ~100-150 |
| `verl/workers/sharding_manager/megatron_vllm.py`                | `patch_vllm_moe_model_weight_loader`    | ~172     |
| `verl/workers/megatron_workers.py`                              | `ActorRolloutRefWorker.init_model`      | ~63      |
| `verl/workers/megatron_workers.py`                              | `generate_sequences`                    | ~282-312 |
| `verl/workers/megatron_workers.py`                              | `update_actor`                          | ~后续      |
| `verl/third_party/vllm/vllm_v_0_6_3/parallel_state.py`          | 双环境初始化                                  | 全文       |
| `verl/third_party/vllm/vllm_v_0_6_3/megatron_weight_loaders.py` | 权重格式转换                                  | 全文       |
| `verl/utils/megatron_utils.py`                                  | `offload_megatron_*`                    | ~240-420 |


# GitHub 直接链接
你可以在以下 URL 直接查看源代码：
- 主实现: https://github.com/volcengine/verl/blob/main/verl/workers/sharding_manager/megatron_vllm.py
- Worker 调用: https://github.com/volcengine/verl/blob/main/verl/workers/megatron_workers.py
- vLLM 适配器: https://github.com/volcengine/verl/blob/main/verl/third_party/vllm/
- Megatron 工具: https://github.com/volcengine/verl/blob/main/verl/utils/megatron_utils.py

## 注意：
- <font color='red'>verl 在 v0.6.0 后已弃用所有 sharding manager</font>，转向新的 BaseRollout 接口和 RemoteBackend 抽象，但核心 NCCL 重分片逻辑仍然保留在新的架构中。